In [ ]:
#importing libraries

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
#loading the dataset

data = pd.read_csv('Titanic-Dataset.csv')

data = data.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])



In [ ]:
#handling missing values and modifying data


data["Age"] = data["Age"].fillna(data['Age'].median())

data["Embarked"] = data["Embarked"].fillna(data['Embarked'].mode()[0])

data['Sex'] = data['Sex'].map({'male': 0, 'female': 1})

data = pd.get_dummies(data, columns=['Embarked'])


data.isnull().sum()




,Survived,Pclass,Sex,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,0.352413,29.361582,0.523008,0.381594,32.204208
std,0.486592,0.836071,0.477990,13.019697,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,0.000000,22.000000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,0.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,1.000000,35.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,1.000000,80.000000,8.000000,6.000000,512.329200


In [ ]:
#preparing the data for training and testing

X = data.drop(columns=['Survived'])
y = data['Survived'].values

y = y - y.min()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


X_train  = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)






In [ ]:
# Model architecture and training

num_classes = len(data['Survived'].unique())

class TitanicNN(nn.Module):
    def __init__(self):
        super(TitanicNN, self).__init__()
        self.fc1 = nn.Linear(X_train.shape[1], 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x
    
model = TitanicNN()
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
y_train = y_train.float().unsqueeze(1)
y_test = y_test.float().unsqueeze(1)

num_epochs = 500

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}")

Epoch [10/500], Loss: 0.6694
Epoch [20/500], Loss: 0.6319
Epoch [30/500], Loss: 0.5851
Epoch [40/500], Loss: 0.5283
Epoch [50/500], Loss: 0.4786
Epoch [60/500], Loss: 0.4493
Epoch [70/500], Loss: 0.4350
Epoch [80/500], Loss: 0.4247
Epoch [90/500], Loss: 0.4160
Epoch [100/500], Loss: 0.4087
Epoch [110/500], Loss: 0.4025
Epoch [120/500], Loss: 0.3971
Epoch [130/500], Loss: 0.3922
Epoch [140/500], Loss: 0.3877
Epoch [150/500], Loss: 0.3836
Epoch [160/500], Loss: 0.3797
Epoch [170/500], Loss: 0.3759
Epoch [180/500], Loss: 0.3724
Epoch [190/500], Loss: 0.3690
Epoch [200/500], Loss: 0.3657
Epoch [210/500], Loss: 0.3625
Epoch [220/500], Loss: 0.3594
Epoch [230/500], Loss: 0.3565
Epoch [240/500], Loss: 0.3535
Epoch [250/500], Loss: 0.3507
Epoch [260/500], Loss: 0.3481
Epoch [270/500], Loss: 0.3456
Epoch [280/500], Loss: 0.3431
Epoch [290/500], Loss: 0.3406
Epoch [300/500], Loss: 0.3381
Epoch [310/500], Loss: 0.3355
Epoch [320/500], Loss: 0.3331
Epoch [330/500], Loss: 0.3306
Epoch [340/500], Lo

In [ ]:
# Evaluating the model 

model.eval()
with torch.no_grad():
    predictions = model(X_test)
    predictions_classes = (predictions > 0.5).float()

    accuracy = (predictions_classes == y_test).float().mean()
    print(f"Test Accuracy: {accuracy.item():.4f}")

for i in range(10):
    pred = int(predictions_classes[i].item())
    actual = int(y_test[i].item())
    print(f"Predicted: {pred}, Real: {actual}")

Test Accuracy: 0.8136
Predicted: 0, Real: 1
Predicted: 0, Real: 0
Predicted: 0, Real: 0
Predicted: 1, Real: 1
Predicted: 0, Real: 1
Predicted: 1, Real: 1
Predicted: 1, Real: 1
Predicted: 0, Real: 0
Predicted: 1, Real: 1
Predicted: 1, Real: 1
